# 05. Uncertainty and outliers

The shaded band around a forecast is easy to over-trust. This notebook shows where Prophet's intervals come from, which settings change them, how to check them against what actually happened (they usually come out too narrow), and how a few bad data points can wreck both the forecast and the band.

- Where the intervals come from
- `interval_width`
- `changepoint_prior_scale` and the width of the future trend band
- `uncertainty_samples`, and switching intervals off
- A small `mcmc_samples` example
- Checking coverage honestly with `result.backtest()`
- Outliers: how they distort the fit and the intervals, and removing them

Previous: [04. Regressors and weather](04_regressors_and_weather.ipynb) | Next: [06. Evaluation and tuning](06_evaluation_and_tuning.ipynb)

In [ ]:
import contextlib
import logging
import time
from collections.abc import Iterator
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from prophet import Prophet, forecast

from helpers import load_example, quiet_logs

quiet_logs()

`forecast()` logs a note when it drops rows, which would land on stderr. This small helper catches such notes and prints them as ordinary output, so the notebook shows them once, where they matter.

In [ ]:
@contextlib.contextmanager
def forecast_notes() -> Iterator[None]:
    """Print what forecast() logs about the data instead of leaving it on stderr."""
    notes: list[str] = []

    def collect(record: logging.LogRecord) -> bool:
        notes.append(record.getMessage())
        return False

    logger = logging.getLogger("prophet.quick")
    logger.addFilter(collect)
    try:
        yield
    finally:
        logger.removeFilter(collect)
        for note in notes:
            print(f"Note from forecast(): {note}")

## 1. Where the intervals come from

Prophet simulates many possible futures and reports the range that contains most of them. Each simulation has two random parts:

- **Observation noise:** a normal error with the standard deviation `sigma_obs` that the fit found. It is the same on every day.
- **Trend change:** future changepoints appear at the same average rate as in the history, with sizes drawn from the fitted changepoint sizes. The further ahead, the more such changes can pile up.

Seasonality and regressor effects are held at their fitted values, so they add no width (unless you use MCMC, below). The Peyton Manning series is a good example: it has plenty of trend changes in its past.

In [ ]:
df = load_example("peyton_manning")
result = forecast(df, "365D", freq="D")  # the series skips some days, so freq is given
fig = result.plot()

The next chart splits the band's width into its parts. `trend_lower` and `trend_upper` in `result.predictions` hold the trend-only band. The dotted line is the width that observation noise alone would give.

In [ ]:
fcst = result.predictions
last_day = df["ds"].max()
z80 = NormalDist().inv_cdf(0.9)
sigma = float(np.mean(result.model.params["sigma_obs"])) * result.model.y_scale

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fcst["ds"], fcst["yhat_upper"] - fcst["yhat_lower"], label="full 80% band")
ax.plot(fcst["ds"], fcst["trend_upper"] - fcst["trend_lower"], label="trend uncertainty alone")
ax.axhline(2 * z80 * sigma, color="k", linestyle=":", label="observation noise alone")
ax.axvline(last_day, color="grey", linestyle="--", linewidth=1)
ax.set_ylabel("Band width (log page views)")
ax.set_title("Width of the 80% interval over time")
ax.legend()
fig.tight_layout()

**Takeaway:** in the history the band is pure observation noise (the trend is known there). After the last day the trend part grows with distance and widens the band. Note what is not in the band: the chance that the model is wrong.

## 2. `interval_width`

`interval_width` is the share of simulated futures the band should contain: 0.8 by default. It rescales the band you already have, so wider values cost nothing but a wider band.

In [ ]:
widths = (0.5, 0.8, 0.95)
by_width = {w: forecast(df, "365D", freq="D", interval_width=w) for w in widths}

fig, ax = plt.subplots(figsize=(10, 4))
for w in reversed(widths):
    ahead = by_width[w].future
    ax.fill_between(ahead["ds"], ahead["yhat_lower"], ahead["yhat_upper"], alpha=0.25, label=f"{w:.0%} interval")
ax.plot(ahead["ds"], ahead["yhat"], color="k", linewidth=1, label="forecast")
ax.set_ylabel("Log page views")
ax.set_title("The same forecast at three interval widths")
ax.legend()
fig.tight_layout()

pd.Series(
    {f"{w:.0%}": (r.future["yhat_upper"] - r.future["yhat_lower"]).mean() for w, r in by_width.items()},
    name="mean band width",
).round(2)

**Takeaway:** the forecast line does not move, only the band. A wider band is not a more careful one; it only claims more. Whether 80% is honest is a question for a backtest, further down.

## 3. `changepoint_prior_scale` and the future trend band

Future trend changes are drawn from the sizes of the changes the model fitted in the past (their mean absolute size is the scale of the draw). A larger `changepoint_prior_scale` lets the fit use bigger changes, so the future band opens up faster. This is the main way a trend setting changes uncertainty, not only the point forecast.

In [ ]:
scales = (0.01, 0.05, 0.5)
by_scale = {s: forecast(df, "365D", freq="D", changepoint_prior_scale=s) for s in scales}

fig, ax = plt.subplots(figsize=(10, 4))
for s, r in by_scale.items():
    ahead = r.future
    ax.plot(np.arange(1, len(ahead) + 1), ahead["yhat_upper"] - ahead["yhat_lower"], label=f"changepoint_prior_scale={s}")
ax.set_xlabel("Days ahead")
ax.set_ylabel("Band width (log page views)")
ax.set_title("Width of the 80% interval by horizon")
ax.legend()
fig.tight_layout()

pd.DataFrame(
    {
        "mean |rate change|": {s: np.abs(r.model.params["delta"]).mean() for s, r in by_scale.items()},
        "band width, day 1": {s: (r.future["yhat_upper"] - r.future["yhat_lower"]).iloc[0] for s, r in by_scale.items()},
        "band width, day 365": {s: (r.future["yhat_upper"] - r.future["yhat_lower"]).iloc[-1] for s, r in by_scale.items()},
    }
).rename_axis("changepoint_prior_scale").round(3)

**Takeaway:** at 0.01 the band is nearly flat, at the default 0.05 it widens gently and at 0.5 it balloons within a year. The table shows why: the width follows the mean size of the fitted rate changes. Do not pick this value to get a band you like; pick it by forecast accuracy (notebook 06) and read the band as a consequence.

## 4. `uncertainty_samples`, and switching intervals off

The band is estimated from simulated futures: `uncertainty_samples=1000` by default. Fewer samples are faster to predict but make the band jitter from one call to the next. The table predicts the same dates ten times for each setting and reports the spread of the band width on the last day.

In [ ]:
def band_stability(res, repeats: int = 10) -> dict[str, float]:
    """Mean and spread of the last-day band width over repeated predictions, plus the time per call."""
    dates = res.predictions[["ds"]]
    widths, started = [], time.perf_counter()
    for _ in range(repeats):
        out = res.model.predict(dates)
        widths.append(float(out["yhat_upper"].iloc[-1] - out["yhat_lower"].iloc[-1]))
    return {
        "mean width": np.mean(widths),
        "spread (std)": np.std(widths),
        "seconds per predict": (time.perf_counter() - started) / repeats,
    }


samples = (50, 200, 1000)
pd.DataFrame(
    {n: band_stability(forecast(df, "365D", freq="D", uncertainty_samples=n)) for n in samples}
).T.rename_axis("uncertainty_samples").round(3)

**Takeaway:** 1,000 samples gives a steady band for a modest cost; 50 is quick but jumps around. When you do not need a band at all, set `uncertainty_samples=0`: Prophet skips the simulation, the forecast has no interval columns and the chart draws none.

In [ ]:
no_bands = forecast(df, "365D", freq="D", uncertainty_samples=0)
print("Columns:", list(no_bands.forecast.columns))
print("Backtest coverage:", no_bands.backtest("90D").metrics.coverage)
fig = no_bands.plot()

## 5. A small `mcmc_samples` example

By default Prophet finds one best-fit set of parameters (MAP), so seasonality is a fixed curve. With `mcmc_samples` it samples the posterior instead, and the seasonal components get their own uncertainty. It is much slower on real data, so this runs on the first six months of Peyton Manning only. `forecast()` cannot pass fit options such as `show_progress=False`, and MCMC would draw four progress bars, so this section uses `Prophet` directly.

In [ ]:
short = df.iloc[:181]

started = time.perf_counter()
result_map = forecast(short, "60D", freq="D")
map_seconds = time.perf_counter() - started

started = time.perf_counter()
m = Prophet(mcmc_samples=300)
m.fit(short, show_progress=False)
fcst_mcmc = m.predict(m.make_future_dataframe(periods=60))
mcmc_seconds = time.perf_counter() - started

band_map = result_map.future["yhat_upper"] - result_map.future["yhat_lower"]
band_mcmc = fcst_mcmc.tail(60)["yhat_upper"] - fcst_mcmc.tail(60)["yhat_lower"]
pd.DataFrame(
    {"MAP (default)": [map_seconds, band_map.mean()], "MCMC, 300 samples": [mcmc_seconds, band_mcmc.mean()]},
    index=["seconds", "mean band width"],
).round(2)

In [ ]:
fig = result_map.plot_components()

In [ ]:
fig = m.plot_components(fcst_mcmc)

**Takeaway:** under MAP the weekly profile is a single line and the trend has no band in the history; under MCMC both carry a band, and the forecast band is wider. Here MCMC costs seconds for 181 rows, but expect minutes on years of data. `m.predictive_samples(future)` returns the raw simulated futures if you want to compute your own quantiles.

## 6. Checking coverage honestly with `result.backtest()`

A band is only useful if it is right about as often as it claims. `result.backtest(horizon)` holds out the end of the history, refits on the rest, forecasts the held-out period and reports `coverage`: the share of actual values inside the band. For an 80% band, 0.8 is the target. Start with five example datasets at two widths.

In [ ]:
pedestrians = load_example("pedestrians")
cases = {
    "peyton_manning": (df, "D", "365D"),
    "wp_log_r": (load_example("wp_log_r"), "D", "365D"),
    "air_passengers": (load_example("air_passengers"), "MS", "24 months"),
    "retail_sales": (load_example("retail_sales"), "MS", "24 months"),
    "pedestrians (to 2019)": (pedestrians[pedestrians["ds"] < "2020-01-01"], "D", "90D"),
}


def coverage_of(data: pd.DataFrame, freq: str, horizon: str, width: float) -> float:
    """Share of held-out values that fall inside an interval of the given width."""
    run = forecast(data, horizon, freq=freq, interval_width=width).backtest(horizon)
    return run.metrics.coverage


pd.DataFrame(
    {
        name: {"holdout": horizon, "coverage at 80%": coverage_of(data, freq, horizon, 0.8),
               "coverage at 95%": coverage_of(data, freq, horizon, 0.95)}
        for name, (data, freq, horizon) in cases.items()
    }
).T

**Takeaway:** Peyton Manning and the pre-2020 pedestrians land near 80%, but `wp_log_r` is a little short and Air Passengers and Retail Sales cover only about half of what happened. At 95%, four of the five fall short. Short holdouts of 24 months make single numbers noisy, yet the direction is clear: the intervals assume the future behaves like the past, and these series changed.

One holdout is a single roll of the dice. For a steadier picture, repeat the backtest from several points in the Peyton Manning history (each row below refits on the data up to that date and holds out its last 90 days), at four interval widths.

In [ ]:
origins = list(range(1000, len(df) - 90, 300))
levels = (0.8, 0.9, 0.95, 0.99)

calibration = pd.DataFrame(
    {
        f"{w:.0%}": [
            coverage_of(df.iloc[:end], "D", "90D", w) for end in origins
        ]
        for w in levels
    },
    index=pd.Index([df["ds"].iloc[end - 1].date() for end in origins], name="data ends"),
)
calibration.round(2)

In [ ]:
achieved = calibration.mean()
print("Mean coverage across the origins:")
print(achieved.map("{:.0%}".format).rename("achieved").to_frame().T.to_string())

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(levels, achieved.to_numpy(), "o-", label="achieved")
ax.plot([0.75, 1.0], [0.75, 1.0], "k:", label="perfect calibration")
ax.set_xlabel("Nominal interval width")
ax.set_ylabel("Share of actual values inside")
ax.set_title("Nominal against achieved coverage")
ax.legend()
fig.tight_layout()

**Takeaway:** for this series the bands under-cover at every level, and by a lot for the default 80% (about 60% achieved, with large swings from one origin to the next). A nominal 95% band delivered roughly the 80% that was asked for. Raising `interval_width` is a blunt but honest fix: choose the nominal width at which backtests reach the coverage you need, on your own data (notebook 06 covers cross-validation over many cutoffs). Until you have checked, treat the bands as optimistic.

## 7. Outliers: how they distort the intervals

Prophet is robust to a lot, but not to a block of bad values, such as a logging outage or a pasted-over month. Two example series have exactly that. Plot the raw data first and look; choosing the window is a judgement call.

In [ ]:
df1 = load_example("wp_log_r_outliers1")
bad1 = df1["ds"].between("2010-01-02", "2010-12-31")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df1["ds"], df1["y"], ".", markersize=2)
ax.axvspan(pd.Timestamp("2010-01-02"), pd.Timestamp("2010-12-31"), color="tab:red", alpha=0.15, label="suspect block")
ax.set_title("wp_log_r_outliers1: a block of constant values through 2010")
ax.legend(loc="lower right")
fig.tight_layout()

Those values are not real traffic. Prophet fits them with trend changes, so the history looks explained but the model now believes the trend swings wildly. Forecast three years ahead:

In [ ]:
result_dirty = forecast(df1, "1096D", freq="D")
fig = result_dirty.plot()

The best fix is to remove them: set `y` to NaN for the affected dates. Prophet has no problem with missing values, and `forecast()` drops those rows (it tells you how many).

In [ ]:
df1_clean = df1.assign(y=df1["y"].mask(bad1))

with forecast_notes():
    result_clean = forecast(df1_clean, "1096D", freq="D")
fig = result_clean.plot()

In [ ]:
def interval_summary(res) -> pd.Series:
    """Noise level, band width on the first and last forecast day, and the last forecast value."""
    ahead = res.future
    width = ahead["yhat_upper"] - ahead["yhat_lower"]
    return pd.Series({
        "sigma_obs (log units)": float(np.mean(res.model.params["sigma_obs"])) * res.model.y_scale,
        "band width, day 1": width.iloc[0],
        "band width, last day": width.iloc[-1],
        "forecast, last day": ahead["yhat"].iloc[-1],
    })


pd.DataFrame({"with outliers": interval_summary(result_dirty), "outliers removed": interval_summary(result_clean)}).round(2)

**Takeaway:** the centre line barely changes, but the band at the end of three years shrinks from enormous to sensible. The outliers were fitted as trend changes, and the uncertainty model then expects more of the same. Removing them tells the model they never happened.

## 8. Outliers: when they also move the forecast

The second series has a short run of extreme lows in June 2015. Here the damage reaches the forecast itself.

In [ ]:
df2 = load_example("wp_log_r_outliers2")
bad2 = df2["ds"].between("2015-06-02", "2015-06-29")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(df2["ds"], df2["y"], ".", markersize=2)
ax.axvspan(pd.Timestamp("2015-06-02"), pd.Timestamp("2015-06-29"), color="tab:red", alpha=0.15, label="suspect block")
ax.set_title("wp_log_r_outliers2: a few weeks of extreme lows in June 2015")
ax.legend(loc="lower left")
fig.tight_layout()

In [ ]:
result_dirty2 = forecast(df2, "1096D", freq="D")
fig = result_dirty2.plot()

In [ ]:
df2_clean = df2.assign(y=df2["y"].mask(bad2))

with forecast_notes():
    result_clean2 = forecast(df2_clean, "1096D", freq="D")
fig = result_clean2.plot()

For reference, the same page's original series without any injected outliers, forecast the same way. The chart overlays the three forecasts.

In [ ]:
reference = forecast(load_example("wp_log_r"), "1096D", freq="D")
runs = {"with outliers": result_dirty2, "outliers removed": result_clean2, "original, no outliers": reference}

fig, ax = plt.subplots(figsize=(10, 4))
for label, res in runs.items():
    ax.plot(res.future["ds"], res.future["yhat"], label=label)
ax.set_ylabel("Log page views")
ax.set_title("Three-year forecasts of the same series")
ax.legend()
fig.tight_layout()

pd.DataFrame({label: interval_summary(res) for label, res in runs.items()}).round(2)

**Takeaway:** four weeks of bad values leak into the yearly seasonality, so every June in the forecast gets a deep dip and the whole forecast sits about 0.8 log units too low on average (roughly half the page views). The noise estimate more than doubles, so the band starts too wide; the dirty band then stays flat while the clean one widens with the horizon, as it should. Removing the bad weeks brings the forecast back to where the clean series puts it. If an event is real and recurring, model it (holidays, notebook 03; a regressor, notebook 04) rather than deleting it.

## Next

[06. Evaluation and tuning](06_evaluation_and_tuning.ipynb) goes further with backtests: cross-validation across many cutoffs, accuracy metrics by horizon, and choosing hyperparameters.